# Financial Sentiment Study — V2

**Use this clean notebook in Colab. Do not paste in repair cells from the old version.**

[Open V2 in Google Colab](https://colab.research.google.com/github/AMurtadaa/Stock-Monitor/blob/class/financial-sentiment-study/class_project/Financial_Sentiment_Study_V2.ipynb)

GitHub shows the notebook; **Colab runs it**. Select **Runtime → Change runtime type → GPU**.
Run sections 1–4 in order. Section 4 must explicitly print **SMOKE TEST PASSED** before
section 5 allows full training. Every command prints START/PASS/FAILED and a heartbeat
after 15 seconds without new output. Saved Drive data and checkpoints are reused.
This notebook does not modify your deployed application.


## 1. Load the current workflow and preserve Drive artifacts

First stop any cell still running in the old notebook. This uses a new code checkout
and a new isolated environment, while keeping your existing Drive dataset/checkpoints.
No restart, deletion or manual package patch is needed. Authorize Drive when prompted.


In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

print("FINANCIAL SENTIMENT WORKFLOW V2 — loading", flush=True)
USE_DRIVE = True
repo = Path("/content/stock-monitor-class-v2")
if not repo.exists():
    print("Downloading the current class branch...", flush=True)
    downloaded = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "class/financial-sentiment-study",
         "https://github.com/AMurtadaa/Stock-Monitor.git", str(repo)],
        capture_output=True, text=True,
    )
    print(downloaded.stdout + downloaded.stderr, flush=True)
    downloaded.check_returncode()
else:
    print("Updating the V2 code checkout without resetting local changes...", flush=True)
    updated = subprocess.run(
        ["git", "-C", str(repo), "pull", "--ff-only"],
        capture_output=True, text=True,
    )
    print(updated.stdout + updated.stderr, flush=True)
    updated.check_returncode()
os.chdir(repo)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("SOURCE COMMIT:", commit, flush=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    saved = Path("/content/drive/MyDrive/CSCI390_Financial_Sentiment/class_artifacts")
    saved.mkdir(parents=True, exist_ok=True)
    local = repo / "class_artifacts"
    if not local.exists() and not local.is_symlink():
        local.symlink_to(saved, target_is_directory=True)
    if local.resolve() != saved.resolve():
        raise RuntimeError("Artifacts exist outside Drive. Export before changing storage.")
spec = importlib.util.spec_from_file_location(
    "class_notebook_runtime_v2", repo / "class_project/notebook_runtime.py"
)
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
assert runtime.WORKFLOW_VERSION == "2.0", "Open the current V2 notebook."
app = runtime.NotebookSession(repo)
print("WORKFLOW:", runtime.WORKFLOW_VERSION, flush=True)
print("ARTIFACTS:", app.artifacts.resolve(), flush=True)
print("SECTION 1 PASSED. Continue to setup.", flush=True)


## 2. One setup cell, with visible progress

This creates pip if absent, installs dependencies, checks imports/GPU and runs tests.
Installation may take several minutes. A heartbeat means the command is still running,
not that it has passed. Errors are printed in full and saved to `class_artifacts/setup_logs`.
Interrupting a command terminates its subprocess group in Colab instead of leaving
an invisible training/install process behind. Wait for **SETUP PASSED**.


In [ ]:
app.setup("/content/stock-class-env-v2", require_gpu=True)


## 3. Prepare and inspect the human-labeled data
Existing frozen splits are reused. Wait for PASS.


In [ ]:
app.run("prepare")
import json

manifest = json.loads((app.artifacts / "dataset/manifest.json").read_text())
print("SPLITS:", manifest["split_sizes"], flush=True)
print("LABEL COUNTS:", manifest["class_counts"], flush=True)


## 4. Smoke test — require an explicit result

This checks two training steps, saving, reloading and development-only evaluation.
It checks actual output files after the commands finish. Reused successful runs still
produce a visible result. Wait for **SMOKE TEST PASSED**. Low smoke accuracy is expected.


In [ ]:
app.smoke()


## 5. Three full training runs
This is gated on section 4. Keep the runtime connected; checkpoints go to Drive.


In [ ]:
app.full_training()


## 6. Held-out evaluation
Accuracy means the fraction correct. Macro-F1 treats each class equally. FinBERT is a reference only because it saw Financial PhraseBank during its original training. Inspect the 15 review examples and add your explanations.


In [ ]:
app.run("evaluate", "--finbert")
metrics = json.loads(Path("class_artifacts/evaluation/full/metrics.json").read_text())
print(json.dumps(metrics["three_seed_summary"], indent=2))
from IPython.display import Image, display

display(Image(filename="class_artifacts/evaluation/full/seed-42_confusion.png"))
display(Image(filename="class_artifacts/evaluation/full/seed-42_curves.png"))

## 7. Historical news for the financial comparison
This is separate from sentiment training. We need actual timestamps and later prices to test whether sentiment helps predictions.

First inspect existing inputs. This fresh fork contains source, not your other machine's private database. If you already have the FNSPID archive, set `NEWS_CSV` to its path in Drive. Otherwise the default URL streams the public ~23 GB archive once, retaining only 12 stocks. **This can take a long time.** It prints progress; do not repeat after `headlines.csv` exists. No credentials or generated sentiment are used.


In [ ]:
app.run("inspect-data")
NEWS_CSV = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/nasdaq_exteral_data.csv"
# If you have the archive in Drive, replace the URL above with that file path.
if not Path("class_artifacts/financial/headlines.csv").exists():
    app.run("filter-news", "--csv", NEWS_CSV)
print(Path("class_artifacts/financial/news_manifest.json").read_text())
app.run("score-news")

## 8. Prices, filing-dated fundamentals, and the controlled comparison
SEC requires a descriptive contact User-Agent. Enter your own email only in the prompt; it is used for SEC requests and isn't saved in the class report. Free SEC requests may be blocked from some Colab networks; if so, stop and share the error so we can prepare the snapshot locally.

The comparison trains the same LightGBM model twice, changing only whether it receives sentiment. It stops if news coverage is too weak or dates are missing. It does not submit trades.


In [ ]:
from getpass import getpass

contact = getpass("SEC contact email (for request identification only): ").strip()
if not Path("class_artifacts/financial/snapshot/manifest.json").exists():
    app.run("snapshot", "--sec-user-agent", f"CSCI390Research/1.0 ({contact})")
del contact
app.run("compare")
print("Read coverage first:")
print(Path("class_artifacts/financial/coverage.json").read_text())

## 9. Prepare and export results before opening the demo
The report generator inserts actual saved metrics into a draft. It cannot write your personal interpretation for you. Download the archive before ending the session; Drive holds a second copy when enabled. Send `submission/results_summary.json` back for review. Do not send a private app database or API keys.


In [ ]:
app.run("writeup")
app.run("export")
from google.colab import files

files.download("class_artifacts/exports/class_training_artifacts.zip")

## 10. Interactive demonstration
This cell remains running while the demo is open. `--share` creates a temporary public link. Only curated examples and saved class metrics are shown. Try a positive, a negative, a neutral and a contrasting sentence. Stop this cell to close the demo.


In [ ]:
app.run("demo", "--finbert", "--share")